# Eksperimen Preprocessing - Insurance Cost Prediction
**Nama:** Ari Setiawan
**Dataset:** Insurance Cost Prediction
**Tujuan:** Melakukan eksplorasi dan preprocessing data sebelum pelatihan model

## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from joblib import dump
import os
import warnings
warnings.filterwarnings('ignore')

print('Library berhasil diimport!')

## 2. Data Loading

In [ ]:
df = pd.read_csv('../insurance_raw.csv')
print(f'Shape dataset: {df.shape}')
df.head()

In [ ]:
print('Informasi Dataset:')
df.info()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
print('Statistik Deskriptif:')
df.describe()

In [ ]:
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# Distribusi kolom numerik
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df['age'].hist(ax=axes[0,0], bins=20, color='steelblue', edgecolor='white')
axes[0,0].set_title('Distribusi Age')

df['bmi'].hist(ax=axes[0,1], bins=20, color='steelblue', edgecolor='white')
axes[0,1].set_title('Distribusi BMI')

df['children'].hist(ax=axes[1,0], bins=6, color='steelblue', edgecolor='white')
axes[1,0].set_title('Distribusi Children')

df['charges'].hist(ax=axes[1,1], bins=20, color='steelblue', edgecolor='white')
axes[1,1].set_title('Distribusi Charges (Target)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi kolom kategorikal
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['sex'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Sex')

df['smoker'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Distribusi Smoker')

df['region'].value_counts().plot(kind='bar', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Distribusi Region')

plt.tight_layout()
plt.show()

In [ ]:
# Korelasi
plt.figure(figsize=(8, 6))
numeric_df = df.select_dtypes(include=['float64', 'int64'])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Heatmap Korelasi Fitur Numerik')
plt.tight_layout()
plt.show()

In [ ]:
# Pengaruh smoker terhadap charges
plt.figure(figsize=(8, 5))
sns.boxplot(x='smoker', y='charges', data=df, palette='Set2')
plt.title('Pengaruh Smoking terhadap Charges')
plt.show()

## 4. Preprocessing

In [ ]:
# Identifikasi fitur
target_column = 'charges'
numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

print(f'Target       : {target_column}')
print(f'Numerik      : {numeric_features}')
print(f'Kategorikal  : {categorical_features}')

In [ ]:
# Buat pipeline preprocessing
numeric_transformer = Pipeline(steps=[
    ('scaler', MinMaxScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

print('Pipeline preprocessing berhasil dibuat!')

In [ ]:
# Split data
X = df.drop(columns=[target_column])
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')

In [ ]:
# Fit dan transform
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'X_train setelah preprocessing: {X_train_processed.shape}')
print(f'X_test setelah preprocessing : {X_test_processed.shape}')

In [ ]:
# Simpan hasil preprocessing
os.makedirs('insurance_preprocessing', exist_ok=True)

train_df = pd.DataFrame(X_train_processed).assign(target=y_train.reset_index(drop=True))
test_df  = pd.DataFrame(X_test_processed).assign(target=y_test.reset_index(drop=True))

train_df.to_csv('insurance_preprocessing/insurance_train_preprocessed.csv', index=False)
test_df.to_csv('insurance_preprocessing/insurance_test_preprocessed.csv',  index=False)

# Simpan pipeline
dump(preprocessor, 'preprocessor.joblib')

print('Dataset preprocessed berhasil disimpan!')
print(f'Train: {train_df.shape}')
print(f'Test : {test_df.shape}')

In [ ]:
print('Preview train dataset preprocessed:')
train_df.head()